# Homework 2 — Vector Search

Embed the course lessons and search them by similarity, then compare keyword,
vector and hybrid search. Embeddings come from the ONNX model
`Xenova/all-MiniLM-L6-v2`. Same 72 pages as homework 1 (commit `8c1834d`).

In [1]:
import numpy as np
from embedder import Embedder

embedder = Embedder()

## Q1 — Embedding a query

384-dim normalized vector; read its first value.

In [2]:
QUERY = "How does approximate nearest neighbor search work?"
v = embedder.encode(QUERY)
len(v), round(float(v[0]), 4)

(384, -0.0206)

## Load the lessons

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)

72

## Q2 — Cosine similarity

Normalized vectors, so the dot product is the cosine similarity.

In [4]:
page = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)
page_vector = embedder.encode(page["content"])
round(float(page_vector.dot(v)), 4)

0.3611

## Q3 — Search by hand

Chunk the pages, embed every chunk into `X`, score with `X.dot(v)`.

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
X = embedder.encode_batch([c["content"] for c in chunks])

scores = X.dot(v)
chunks[int(np.argmax(scores))]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4 — Vector search with minsearch

In [6]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

q4_vec = embedder.encode("What metric do we use to evaluate a search engine?")
vindex.search(q4_vec, num_results=5)[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

## Q5 — Text vs vector

Which file the vector top 5 has that the text top 5 misses.

In [7]:
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"
q5_vec = embedder.encode(q5_query)

vec_files = [r["filename"] for r in vindex.search(q5_vec, num_results=5)]
txt_files = [r["filename"] for r in text_index.search(q5_query, num_results=5)]

[f for f in vec_files if f not in txt_files]

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

## Q6 — Hybrid search (RRF)

Merge both ranked lists by position, `k = 60`.

In [8]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


q6_query = "How do I give the model access to tools?"
q6_vec = embedder.encode(q6_query)

vector_results = vindex.search(q6_vec, num_results=5)
text_results = text_index.search(q6_query, num_results=5)

rrf([vector_results, text_results])[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'

## Answers

| Q | Answer |
|---|--------|
| Q1 | -0.02 |
| Q2 | 0.37 |
| Q3 | `02-vector-search/lessons/07-sqlitesearch-vector.md` |
| Q4 | `04-evaluation/lessons/05-search-metrics.md` |
| Q5 | `02-vector-search/lessons/08-pgvector.md` |
| Q6 | `01-agentic-rag/lessons/13-function-calling.md` |